# Stained-Glass Crisp Demo

Voronoi cells are colored, streaked, and bordered with rounded lead lines. This notebook shows the build-up.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math_vis import seeds_relaxed, cell_colors, glass_crisp, make_cmap, finish

SEED = 4
W, H = 360, 460
N_CELLS = 160
LLOYD = 4
rng = np.random.default_rng(SEED)

In [ ]:
# 1. Generate and relax seeds
seeds = seeds_relaxed(N_CELLS, W, H, rng, iters=LLOYD)
fig, ax = plt.subplots(figsize=(7, 9))
ax.scatter(seeds[:,0], seeds[:,1], s=6, c='#2c3a35')
ax.set_xlim(0,W); ax.set_ylim(0,H); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Lloyd-relaxed Voronoi seeds')
plt.tight_layout(); plt.show()

In [ ]:
# 2. Assign colors to cells
t = cell_colors(seeds, W, H, rng, trend=0.5)

# visualize just the cell assignment
from scipy.spatial import cKDTree
X, Y = np.meshgrid(np.arange(W)+0.5, np.arange(H)+0.5)
P = np.column_stack([X.ravel(), Y.ravel()])
cell = cKDTree(seeds).query(P)[1]
palette = make_cmap(['#3a0ca3', '#4361ee', '#4cc9f0', '#80ffdb', '#ffd166', '#ef476f', '#d62828'])
flat = (palette(t)[cell, :3] * 255).reshape(H, W, 3).astype(np.uint8)

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(flat, origin='lower'); ax.axis('off')
ax.set_title('Flat cells from color recipe')
plt.tight_layout(); plt.show()

In [ ]:
# 3. Build the image stage by stage
img, stages = glass_crisp(
    seeds, t, palette, W, H, rng=rng,
    warp=0.04,
    lead_px=3,
    round_px=2.5,
    streak=0.55,
    streak_light=0.45,
    crack_alpha=0.10,
    contour_alpha=0.12,
    stages=True
)

fig, axes = plt.subplots(2, 3, figsize=(14, 18))
for (name, img), ax in zip(stages.items(), axes.ravel()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 4. Final render at print resolution
final = finish(stages['final'], name='stained_glass_crisp.png', grain=2, vignette=0.35, display=True)